# Communication, Shared State and Collaboration

**Prerequisites:** `02_delegation_and_supervision.ipynb`

Notebook 2 left three things broken, and they are the same thing seen three times.

- The supervisor asked engineering about "this serial" and **left the serial out**. A whole agent run went on answering *"I need the specific serial number."*
- Support replied *"no other batches are affected"*. That string had nowhere to say how sure it was or what it had actually checked, so the supervisor passed it on to a journalist.
- Engineering worked out that a unit outside the recall carried the recalled cell lot. **Nobody else could see that**, so the next customer with the same problem cost another engineering run to find out again.

What a message can carry, what the team can see, and what happens when several agents answer at once: all three are about the shapes between agents rather than the agents themselves.

**By the end you'll have:**
- Replied with a form instead of a paragraph, by giving the agent a `submit` tool, and seen what the fields let the program do that prose cannot
- Made a missing serial number a rejected call instead of a wasted agent run
- Put what the team has established on a shared board, read from each agent's context and written through a tool that knows who owns which field
- Watched two agents write the same field, and fixed it by saying who owns it
- Asked three specialists the same question at once, and aggregated their answers by majority, by weight, and by who actually checked something

## Setup

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(HERE.parent / chapter))

import json
from typing import Annotated

import aggregate
import recall
from messages import Response, answer, form_tool
from meter import Meter
from pydantic import ValidationError
from registry import Registry
from state import Shared, on_board
from supervisor import Agent, as_tool
from tools import Tool

def specialist(name, tools=()):
    return Agent(name, f"{recall.PERSONAS[name]}\n\n{recall.BRIEF}", recall.DESCRIPTIONS[name],
                 tools=Registry(list(tools)))

support = specialist("support", [recall.lookup_serial])
engineer = specialist("engineer", [recall.cell_lot, recall.test_results])
lawyer = specialist("lawyer")
pr = specialist("pr")

REPORT = ("A customer writes: 'My Aria 2 got really hot while charging last night and the case has bulged a bit. "
          "Serial AR2-2409-01544. I'm fine, just worried.'")
costs = {}

## A reply with a shape

In notebook 2 a specialist's reply was whatever text it wrote. The supervisor could read it, and so could we, but nothing else could be done with it: no field to check, nothing to route on, nothing that says "this part is a guess".

The fix is not to parse the prose afterwards. It is to let the agent finish by **filling in a form**, which is an ordinary tool call — the mechanism chapter 1 built and chapter 3 wrapped up as `Tool`. `messages.py` defines the form as a Pydantic model, and `Tool(args=...)` takes a model written by hand, so the fields and their descriptions are exactly what the model is shown:

In [2]:
print(json.dumps(form_tool(Response, []).schema, indent=1))

{
 "type": "function",
 "function": {
  "name": "submit",
  "description": "Hand in your answer. Call this exactly once, when you are done.",
  "parameters": {
   "description": "What an agent hands back. Every field is one the string in notebook 2 had nowhere to put.",
   "properties": {
    "answer": {
     "description": "Your answer, in full. This is what the person asking will act on.",
     "type": "string"
    },
    "confident": {
     "description": "True only if your tools or the brief support every part of the answer.",
     "type": "boolean"
    },
    "visibility": {
     "description": "external if this may be sent to a customer, journalist or regulator as it stands; internal if any of it must stay inside the company.",
     "enum": [
      "internal",
      "external"
     ],
     "type": "string"
    },
    "unknowns": {
     "default": "",
     "description": "Anything you could not establish. Empty if there is nothing.",
     "type": "string"
    },
    "ask_next": {


`submit` stops the agent's loop the moment it is called, through the same `cancel` hook chapter 2 uses for abandoned branches and notebook 2 used for handoffs.

The same request, asked both ways: the notebook 2 way, and with the form.

In [3]:
REQUEST = ("A customer reports that unit AR2-2409-01544 got hot while charging and the case has bulged. "
           "Is this unit affected by the battery fault?")

with Meter() as meter:
    as_string = as_tool(engineer)(request=REQUEST)          # notebook 2: whatever it wrote
    as_form = answer(engineer, REQUEST)                     # this notebook: the form it filled in
costs["both ways"] = meter.summary()

print("=== as a string (notebook 2)\n" + as_string)
print("\n=== as a form")
for field, value in as_form.model_dump().items():
    print(f"  {field:<11} {value}")
print("\n=== what the asking agent sees\n" + as_form.render())
print("\n", costs["both ways"])

=== as a string (notebook 2)
- Unit **AR2-2409-01544** is **not** in the affected batch (AR2‑2408‑00001 to AR2‑2408‑03200).  
- Therefore it is **not confirmed** to have the separator defect that caused overheating.

=== as a form
  answer      - Serial AR2-2409-01544 is NOT in the recalled AR2‑2408 batch (00001‑03200).
- The unit uses cells from lot VC‑7731, the same lot as the faulty batch.
- 400 units from other batches (including this lot) were stress‑tested and showed no fault.
- Therefore, no evidence that this particular unit is affected. It is not covered by the current recall.
- If the customer is concerned, advise them to stop charging and contact support for a safety check, but it is not part of the identified defect.

  confident   True
  visibility  internal
  unknowns    
  ask_next    

=== what the asking agent sees
- Serial AR2-2409-01544 is NOT in the recalled AR2‑2408 batch (00001‑03200).
- The unit uses cells from lot VC‑7731, the same lot as the faulty batch.
- 400

This is one run of each, not a comparison of quality. Both answers say roughly the same thing, and both are wrong in the same way: the unit really was built from lot VC‑7731, and "no evidence this unit is affected" is a poor reading of that. **The form does not make an agent right.** What it changes is what the program around the agent can do:

- `visibility: internal` — this answer is not to be sent to the customer as it stands. The string version said nothing on the subject, and notebook 2's supervisor forwarded exactly that kind of answer to a journalist.
- `confident: True` and an empty `unknowns` — the agent is claiming it settled the question. That claim is now a field, so it can be checked against what the agent actually did. Here it should not have been `True`, which is a thing you can *see* rather than have to infer from prose.
- `ask_next` — empty here, but when it is filled the supervisor has a routing instruction that came from the specialist that knows, rather than from its own guess.

Every field is self-reported. A form is not a check on the answer; it is a place for the answer to say things prose has nowhere to put, and something a program can read without another model call.

## A request with a shape

The other direction. Notebook 2's `ask_engineer` took one free-text `request`, and the supervisor wrote *"Assess whether overheating during charging with a 65 W charger is a known defect for this serial"* — without the serial. Engineering had to spend a run asking for it.

A tool's arguments are checked before the function runs, and `Tool` builds that check from the signature. So make the serial a parameter:

In [3]:
def ask_engineer(
    request: Annotated[str, "What you need engineering to work out."],
    serial: Annotated[str, "The unit's serial number, e.g. AR2-2409-01544. Required: engineering "
                           "cannot look up a cell lot without it."],
) -> str:
    """Hardware engineering. Whether a unit could have the battery fault, and how far it reaches."""
    return answer(engineer, f"{request}\nSerial: {serial}").render()

engineering_tool = Tool(ask_engineer)
print(json.dumps(engineering_tool.schema["function"]["parameters"], indent=1))

try:
    engineering_tool.validate({"request": "Is this serial affected?"})
except ValidationError as refused:
    print("\nthe notebook 2 call, now refused before anything runs:")
    print(refused)

{
 "additionalProperties": false,
 "properties": {
  "request": {
   "description": "What you need engineering to work out.",
   "type": "string"
  },
  "serial": {
   "description": "The unit's serial number, e.g. AR2-2409-01544. Required: engineering cannot look up a cell lot without it.",
   "type": "string"
  }
 },
 "required": [
  "request",
  "serial"
 ],
 "type": "object"
}

the notebook 2 call, now refused before anything runs:
1 validation error for ask_engineer_args
serial
  Field required [type=missing, input_value={'request': 'Is this serial affected?'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


The call is refused by the tool's own argument check, before any agent runs. In notebook 2 the same mistake cost a whole engineering run and a round trip, and the supervisor only found out because the specialist wrote back asking for the serial.

This is worth contrasting with the form above. The reply's fields are **self-reported** — the agent fills them in and can be wrong. The request's fields are **enforced** — the call does not happen at all without them. Both are shapes on a message; only one is a guarantee.

## What the team has established

Some things are neither a message nor a memory. The cell lot behind this customer's unit is not worth keeping after the case closes, and it is not something one agent tells another once: it is the state of the case, and everyone who touches the case needs it.

So it goes on a board. Three decisions are made when the team is wired up, not by the agents:

- **`fields`** — what the case has room for, and what each field means.
- **`owner`** — who may write each field. A field two agents can both set is a field neither can trust.
- **`readers`** — who may see it. The lawyer's note about legal exposure is not support's business.

Reading is by injection: the board renders into the agent's context. Writing is a tool bound to the agent's own name, so a write has an author and ownership can be checked.

In [4]:
board = Shared(
    fields={
        "serial": "the unit this case is about",
        "in_recall": "whether the serial is on the recall list",
        "cell_lot": "which battery cell lot the unit was built from",
        "risk": "whether this unit could have the recall fault, and how sure we are",
        "customer_reply": "what was sent to the customer",
        "legal_risk": "exposure if this turns out to reach beyond the recalled batch",
    },
    owner={"in_recall": "support", "cell_lot": "engineer", "risk": "engineer",
           "customer_reply": "support", "legal_risk": "lawyer"},
    readers={"legal_risk": ["lawyer", "supervisor"]},
)

board.set("supervisor", "serial", "AR2-2409-01544")
board.set("lawyer", "legal_risk", "one burned customer already has a lawyer")

print("=== what support sees\n" + board.view("support"))
print("\n=== what the lawyer sees\n" + board.view("lawyer"))
print("\nengineer's tools on this board:", [t.name for t in board.tools("engineer")])
print("pr's tools on this board:      ", [t.name for t in board.tools("pr")])

=== what support sees
What the team has established so far:
- serial: AR2-2409-01544
- in_recall: (not established yet) -- whether the serial is on the recall list
- cell_lot: (not established yet) -- which battery cell lot the unit was built from
- risk: (not established yet) -- whether this unit could have the recall fault, and how sure we are
- customer_reply: (not established yet) -- what was sent to the customer

=== what the lawyer sees
What the team has established so far:
- serial: AR2-2409-01544
- in_recall: (not established yet) -- whether the serial is on the recall list
- cell_lot: (not established yet) -- which battery cell lot the unit was built from
- risk: (not established yet) -- whether this unit could have the recall fault, and how sure we are
- customer_reply: (not established yet) -- what was sent to the customer
- legal_risk: one burned customer already has a lawyer  (set by lawyer)

engineer's tools on this board: ['record']
pr's tools on this board:       []


Support cannot see `legal_risk` at all — not its value, not that the field exists. PR gets no `record` tool, because PR owns nothing here.

Now the case, in two steps. Engineering works out what happened and records it. Then support writes to the customer, with the board in its context and **without anyone asking engineering anything**.

In [6]:
with Meter() as meter:
    finding = on_board(engineer, board).run(
        f"{REPORT}\n\nWork out whether this unit could have the recall fault. Then call `record` twice: once to put cell_lot on the board, and once to put risk there. Do not write to the customer; support does that.")
costs["engineer + board"] = meter.summary()

print(finding)
print("\n=== the board now\n" + board.view("support"))
print("\n=== every write so far")
for entry in board.history:
    print(f"  {entry['author']:<10} {entry['field']:<15} {entry['to']}")
print("\n", costs["engineer + board"])

AR2-2409-01544 was built with battery cells from lot VC-7731.  
Not in recall batch, low risk

=== the board now
What the team has established so far:
- serial: AR2-2409-01544
- in_recall: (not established yet) -- whether the serial is on the recall list
- cell_lot: VC-7731  (set by engineer)
- risk: Not in recall batch, low risk  (set by engineer)
- customer_reply: (not established yet) -- what was sent to the customer

=== every write so far
  supervisor serial          AR2-2409-01544
  lawyer     legal_risk      one burned customer already has a lawyer
  engineer   cell_lot        VC-7731
  engineer   risk            Not in recall batch, low risk

 {'calls': 6, 'prompt': 7291, 'completion': 2217, 'total': 9508, 'seconds': 6}


In [7]:
with Meter() as meter:
    reply = on_board(support, board).run(
        f"{REPORT}\n\nWrite the reply to this customer. Then call `record` twice: once to put in_recall on the board, and once to put customer_reply there.")
costs["support + board"] = meter.summary()

print(reply)
print("\n=== the board now\n" + board.view("support"))
print("\n", costs["support + board"])

Hi. Stop charging your headphones right away. Your serial number AR2‑2409‑01544 is not in the recall batch, so there's no mandatory recall for you. If the case bulges or the headphones feel hot while charging, stop charging again and let us know. You can still use them on battery. If you’d like a replacement or refund, just tell us. We’re sorry for the worry and are here to help.

**record**(field: "in_recall", value: "no")

**record**(field: "customer_reply", value: "Hi. Stop charging your headphones right away. Your serial number AR2‑2409‑01544 is not in the recall batch, so there's no mandatory recall for you. If the case bulges or the headphones feel hot while charging, stop charging again and let us know. You can still use them on battery. If you’d like a replacement or refund, just tell us. We’re sorry for the worry and are here to help.")

=== the board now
What the team has established so far:
- serial: AR2-2409-01544
- in_recall: no  (set by support)
- cell_lot: VC-7731  (set 

Engineering called `cell_lot`, found VC-7731, and wrote two fields. Support then answered the customer and wrote two more. The board ends the case complete: what the unit is, what it was built from, what the risk is, whether it is on the recall list, and what was sent.

**Support never asked engineering anything.** Its reply came out of the board, in three calls. In notebook 2 the same answer cost an extra agent run every time a customer wrote in, and only happened at all once the supervisor was told to ask engineering first.

Two things are worth taking from this beyond that.

**Writing had to be asked for, twice.** An earlier version of these two cells said "record what you find" and "record what you send". Engineering then narrated its findings in prose, and support printed a section headed "Record to shared board" listing the field names — as text, in its answer, with the `record` tool sitting unused. The board stayed empty. The cells now name the fields to write, and both agents write them. Support still prints `**record**(field: "in_recall", value: "no")` in its answer *as well* as calling the tool.

That is the asymmetry to remember: **reading state is reliable because the code does it**, by rendering the board into the prompt. **Writing waits on the agent choosing to call a tool**, so it needs either an explicit instruction or, better, code that writes what it already knows — the reply support just returned did not need support's permission to be recorded.

**A board spreads a judgement as fast as a fact.** `cell_lot: VC-7731` is a fact; engineering looked it up. `risk: Not in recall batch, low risk` is engineering's opinion, written in the same shape, in the same place. Support read the board and wrote *"not in the recall batch, so there's no mandatory recall for you"* — softer than the reply in the notebook's first run, where the risk field read "Possible recall fault; same lot VC-7731". Neither agent did anything wrong. The second one simply inherited the first one's wording, which is the point of the board and also its risk.

## Two writers, one field

`risk` is engineering's field on that board. Here is the same case on a board where nobody said so:

In [8]:
loose = Shared(fields={"risk": "whether this unit could have the recall fault"})
loose.set("support", "risk", "not on the recall list, so no action needed")
loose.set("engineer", "risk", "built from lot VC-7731, the recalled lot: treat as affected")

print("what a reader sees:", loose.values["risk"])
print("\nwhat actually happened:")
for entry in loose.history:
    print(f"  {entry['author']:<10} {entry['from']!r} -> {entry['to']!r}")

print("\nand on the owned board, support's attempt:")
print(" ", board.tools("support")[0](field="risk", value="not on the recall list, so no action needed"))
print("  risk is still:", board.values.get("risk"))

what a reader sees: built from lot VC-7731, the recalled lot: treat as affected

what actually happened:
  support    None -> 'not on the recall list, so no action needed'
  engineer   'not on the recall list, so no action needed' -> 'built from lot VC-7731, the recalled lot: treat as affected'

and on the owned board, support's attempt:
  Refused: risk is engineer's to set, not support's
  risk is still: Not in recall batch, low risk


On the loose board a reader sees one value, engineering's, and it happens to be the right one. It could as easily have been the other way: support wrote first, engineering wrote second, and nothing about the board made that order meaningful. The history shows what really happened — two agents, two different beliefs, one overwriting the other.

On the owned board support's write is refused, and `risk` stays as it is. The refusal is what the agent gets back: *"Refused: risk is engineer's to set, not support's"*, which is a sentence it can act on.

**Ownership is a design decision, not a runtime one.** Someone has to say that the cell lot is engineering's to establish and the customer reply is support's. Once said, a field means the same thing every time it is read, which is the only reason anyone can build on it.

## Three answers to one question

Everything so far has one agent answering at a time. The other shape is several agents answering the *same* question at once, each from its own side of it.

The question is a real one for this recall: the September batch is not recalled, but its first 1,800 units were built from the lot that is. Engineering can look that up. The lawyer knows what the regulator was told. PR knows what a second recall announcement does to the first one. None of them can answer for the others.

`aggregate.py` asks them in parallel, each filling in an `Opinion` form: a position, a reason, and — the field that matters — what they actually checked.

In [9]:
QUESTION = ("A customer's unit AR2-2409-01544 overheated and its case bulged. AR2-2409 is not in the recall. "
            "Should the recall be widened to cover batch AR2-2409? Answer yes or no.")

with Meter() as meter:
    views = aggregate.opinions([engineer, lawyer, pr], QUESTION)
costs["three opinions"] = meter.summary()

print(aggregate.render(views))
print("\n", costs["three opinions"])

engineer says yes (confident)
  because: Cell lot VC‑7731, the same faulty separator defect, is used in AR2‑2409‑01544 and the recall is based on that lot. The defect is not limited to AR2‑2408 batch.
  checked: cell_lot, test_results

lawyer says no (confident)
  because: The brief contains only evidence of overheating in batch AR2-2408; no other batch has been implicated, and the brief does not indicate a decision to widen the recall.
  checked: Incident brief: affected batch AR2-2408 only.

pr says no (confident)
  because: The evidence shows only the AR2-2408 batch has identified defects; AR2-2409 units have not been reported and no fault found in other batches.
  checked: Incident brief details

 {'calls': 5, 'prompt': 5109, 'completion': 1401, 'total': 6510, 'seconds': 4}


In [10]:
print("tally:        ", aggregate.tally(views))
print("majority:     ", aggregate.majority(views))
print("weighted 3-2-1:", aggregate.weighted(views, {"engineer": 3, "lawyer": 2, "pr": 1}))
print("weighted 4-2-1:", aggregate.weighted(views, {"engineer": 4, "lawyer": 2, "pr": 1}))
print("decisive:     ", aggregate.decisive(views, "engineer"))
print("who is where: ", aggregate.disagreement(views))

tally:         {'yes': 1, 'no': 2, 'unsure': 0}
majority:      no
weighted 3-2-1: unsure
weighted 4-2-1: yes
decisive:      ('yes', 'engineer checked: cell_lot, test_results')
who is where:  {'yes': ['engineer'], 'no': ['lawyer', 'pr']}


Every live cell in this notebook is one run, so read the answers as an example rather than a measurement — notebook 2 shows what it takes to claim more than that. This particular split, though, follows from what each agent could reach:

- **Engineering said yes**, having called `cell_lot` and `test_results`.
- **The lawyer and PR said no**, both reasoning from the brief — which says the fault is in batch AR2‑2408 and that other batches tested clean. That is true as far as it goes, and it is exactly what the brief says. It is also out of date: the brief was written before anyone looked up this unit's cell lot.

Now the aggregation, on the same three answers:

| | Result | |
|---|---|---|
| **majority** | no | 2 against 1, and wrong. The two agents in the majority checked the same document as each other |
| **weighted 3‑2‑1** | unsure | engineering's 3 against the other two's 3: a tie. Weights have to be chosen so that they can actually decide |
| **weighted 4‑2‑1** | yes | the same opinions, a different set of weights, the opposite answer |
| **decisive(engineer)** | yes | *because* engineering is confident and named what it checked: `cell_lot, test_results` |

**Majority voting belongs with identical agents**, where chapter 2 used it: ask one agent five times and the majority is its own consensus. Here the voters are not interchangeable. Two of them were never in a position to know, so counting them dilutes the one that was.

Weighting is better and still fragile, as the tie shows: the numbers are yours to pick, and picking them is the whole decision, made in advance and applied to questions you have not seen yet.

`decisive` is the honest version of what you usually mean: **on this question, one agent's evidence settles it, and the others are advice.** It is also the only one of the four that looks at *what an agent checked* rather than at what it concluded — which is possible only because `evidence` is a field on the form. It refuses to decide when that agent is unconfident or checked nothing, which is the right answer often enough to be worth having.

What none of these do is resolve the disagreement in the sense of finding out who is right. `disagreement()` just names the sides. When it matters, that is a question for a human, or for another round where the agents see each other's evidence — which is what the brief itself should become once engineering's finding is in.

## What it cost

In [11]:
print(f"{'':<18}{'calls':>6}{'prompt':>9}{'completion':>12}{'total':>8}{'seconds':>9}")
for name, c in costs.items():
    print(f"{name:<18}{c['calls']:>6}{c['prompt']:>9}{c['completion']:>12}{c['total']:>8}{c['seconds']:>9}")

                   calls   prompt  completion   total  seconds
both ways              3     2982         896    3878        3
three opinions         5     5109        1401    6510        4
engineer + board       6     7291        2217    9508        6
support + board        3     3432        2254    5686        4


- **Nothing here is expensive.** The whole notebook is four live cells and about 20,500 tokens, because most of what it adds is shape, and shape costs nothing to run.
- **The form is free.** `submit` is one more tool in a list the agent was already being sent; filling it in replaces writing the answer as text, it does not come on top.
- **The board pays for itself immediately.** Support answered from it in two calls. Getting the same answer by asking engineering costs a whole extra agent run (notebook 2: three to eight calls), every time.
- **Three opinions cost three agent runs, and one of them decided it.** The other two were paid for to be outvoted, or — the way `decisive` reads it — to be advice. That is worth knowing before asking five agents a question only one can answer.

## Where this leaves us

- **A reply with fields** lets a program act on it: route on `ask_next`, refuse to send when `visibility` is internal, notice a `confident: True` that nothing supports. The fields are self-reported, so they are a place to look, not a guarantee.
- **A request with fields** is a guarantee, because the call is checked before it runs. Notebook 2's missing serial becomes a refusal instead of a wasted agent.
- **A shared board** is the working state of a case: in memory, rendered into each agent's context, and written through a tool that knows who owns what. It is not memory (`04_memory`) and not a log — it is what the team currently believes, and it is what stops the second agent rediscovering the first agent's work.
- **Read is dependable, write is not.** The code injects the board, so reading it always happens. Writing waits on the agent choosing to call a tool: asked to "record what you find", both agents narrated the call in prose instead of making it, and only did it once the fields were named for them.
- **Ownership makes a field mean something.** Without it, the last writer wins and the reader cannot tell that anyone disagreed.
- **Counting specialists is the wrong aggregation.** Majority got it wrong, weights decided it only once they were tuned, and the rule that worked looked at which agent had actually checked something.

Notebook 4 builds one system out of all of this: a roster, a supervisor that routes, forms between agents, a board they work on, and an aggregation step — plus what happens when a worker fails, loops, or hands back something unusable, which this notebook has now produced twice by accident.